# Visual Question Answering sinh văn bản với A-OKVQA

**Deep Learning Midterm Project — 523H0173 & 523H0178**

| Thành phần | Chi tiết |
|-----------|--------|
| **Dataset** | A-OKVQA (Augmented OK-VQA) — Ảnh COCO + Câu hỏi cần tri thức bên ngoài |
| **Data Expansion** | Nhân bản 3× bằng Rationale (~17K → ~51K training samples) để sinh câu dài |
| **Image Encoder** | CNN từ đầu / Pretrained ResNet-18 (Trích xuất lưới không gian 49 vùng) |
| **Question Encoder** | LSTM 2-layer, dropout=0.3, GloVe 300d |
| **Answer Decoder** | LSTM 2-layer sinh văn bản (Generative) |
| **Attention MỚI** | Dual Attention: Text (Bahdanau) + Image (Spatial Attention trên 49 vùng ảnh) |
| **Decoding** | Greedy (khi train) / Beam Search width=5 (khi test) |
| **Metrics** | Accuracy, EM, F1, METEOR, BLEU-1/2/3/4 |

**4 biến thể mô hình để so sánh chéo:**
1. **M1** — CNN (scratch) + LSTM (baseline)
2. **M2** — CNN (scratch) + LSTM + Spatial & Text Attention
3. **M3** — Pretrained ResNet-18 + LSTM
4. **M4** — Pretrained ResNet-18 + LSTM + Spatial & Text Attention (State-of-the-Art)

In [3]:
import os
import sys
import random
import logging
import gc

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import nltk
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)

# Project modules (từ src/)
from src.config import Config
from src.data.preprocessing import normalize_answer, majority_answer, extract_answer, expand_data_with_rationales, classify_question
from src.data.dataset import Vocabulary, AOKVQA_Dataset, collate_fn, PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX
from src.data.glove import download_glove, load_glove_embeddings
from src.models.vqa_model import VQAModel
from src.models.encoder import CNNEncoder, QuestionEncoder
from src.models.decoder import AnswerDecoder
from src.models.attention import BahdanauAttention, SpatialAttention
from src.engine.trainer import train_model, EarlyStopping
from src.engine.evaluator import evaluate_model, evaluate_by_question_type, get_failure_cases
from src.utils.metrics import batch_metrics
from src.utils.helpers import get_device, decode_sequence, set_seed, setup_logging
from src.utils.visualization import (
    plot_training_curves, plot_radar_chart, plot_bar_chart,
    visualize_attention, visualize_attention_overlay,
    plot_confusion_matrix, plot_question_type_analysis,
)

# Load config & setup
cfg = Config.from_yaml("configs/default.yaml")
set_seed(cfg.seed)
logger = setup_logging(cfg.log_dir)
device = get_device() if cfg.device == "auto" else torch.device(cfg.device)

print(f"Device: {device}")
print(f"Seed: {cfg.seed}")
print(f"Batch Size: {cfg.train.batch_size} | Freq Threshold: {cfg.data.freq_threshold}")

Device: mps
Seed: 42
Batch Size: 16 | Freq Threshold: 3


In [4]:
# Cấu hình biến đổi ảnh (Image Transforms)
IMG_SIZE = cfg.data.image_size
transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
transform_eval = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Tải A-OKVQA
print("Downloading A-OKVQA ...")
hf_train = load_dataset(cfg.data.hf_id, split="train")
hf_val   = load_dataset(cfg.data.hf_id, split="validation")

# Xây dựng Vocabulary ưu tiên Rationale
all_questions, all_answers = [], []
for item in tqdm(hf_train, desc="Collecting train text"):
    all_questions.append(item["question"])
    rationales = item.get("rationales", [])
    if rationales:
        all_answers.extend(rationales)
    else:
        all_answers.append(extract_answer(item))
        
for item in tqdm(hf_val, desc="Collecting val text"):
    all_questions.append(item["question"])
    rationales = item.get("rationales", [])
    if rationales:
        all_answers.extend(rationales)
    else:
        all_answers.append(extract_answer(item))

# Threshold đã được tăng lên 3 trong yaml để loại bỏ từ hiếm/rác
question_vocab = Vocabulary(freq_threshold=cfg.data.freq_threshold)
question_vocab.build_vocabulary(all_questions)
answer_vocab = Vocabulary(freq_threshold=cfg.data.freq_threshold)
answer_vocab.build_vocabulary(all_answers)

print(f"\nQuestion vocab: {len(question_vocab):,} tokens")
print(f"Answer vocab  : {len(answer_vocab):,} tokens (Đã lọc nhiễu)")

# Chia split
hf_train_list = list(hf_train)
random.shuffle(hf_train_list)
split_idx = int(len(hf_train_list) * cfg.data.train_ratio)

train_data_raw = hf_train_list[:split_idx]
val_data   = hf_train_list[split_idx:]
test_data  = list(hf_val)

if cfg.data.expand_rationales:
    train_data = expand_data_with_rationales(train_data_raw)
    print(f"\n★ Rationale expansion: {len(train_data_raw):,} → {len(train_data):,} training samples")
else:
    train_data = train_data_raw

train_dataset = AOKVQA_Dataset(train_data, question_vocab, answer_vocab, transform_train)
val_dataset   = AOKVQA_Dataset(val_data,   question_vocab, answer_vocab, transform_eval)
test_dataset  = AOKVQA_Dataset(test_data,  question_vocab, answer_vocab, transform_eval)

# GloVe Embeddings
download_glove()
q_glove_emb = load_glove_embeddings(question_vocab, embed_dim=cfg.model.embed_size)
a_glove_emb = load_glove_embeddings(answer_vocab,   embed_dim=cfg.model.embed_size)


Question vocab: 2,690 tokens
Answer vocab  : 7,490 tokens (Đã lọc nhiễu)

★ Rationale expansion: 14,497 → 43,491 training samples


Loading GloVe: 0it [00:00, ?it/s]

Loading GloVe: 0it [00:00, ?it/s]

In [5]:
BATCH_SIZE = cfg.train.batch_size
_num_workers = 0 if device.type == "mps" else cfg.train.num_workers
_pin_memory = False if device.type == "mps" else cfg.train.pin_memory

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=_num_workers)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=_num_workers)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=_num_workers)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

# Quick Check
images, questions, q_lengths, answers, a_lengths, answer_texts = next(iter(train_loader))
print(f"Images: {images.shape}")
print(f"Questions: {questions.shape}")
print(f"Answers: {answers.shape}")
print(f"Sample Question: {decode_sequence(questions[0].tolist(), question_vocab)}")
print(f"Sample Answer: {decode_sequence(answers[0].tolist(), answer_vocab)}")

Train batches: 2719 | Val batches: 160 | Test batches: 72
Images: torch.Size([16, 3, 224, 224])
Questions: torch.Size([16, 20])
Answers: torch.Size([16, 17])
Sample Question: how many people can cook food here at once
Sample Answer: up to four people could use one of four microwaves on shelves


In [6]:
Q_VOCAB, A_VOCAB = len(question_vocab), len(answer_vocab)

models_dict = {}
for name, variant_cfg in cfg.model_variants.items():
    m = VQAModel(
        Q_VOCAB, A_VOCAB,
        embed_size=cfg.model.embed_size,
        hidden_size=cfg.model.hidden_size,
        num_layers=cfg.model.num_layers,
        dropout=cfg.model.dropout,
        q_pretrained_emb=q_glove_emb,
        a_pretrained_emb=a_glove_emb,
        **variant_cfg,
    )
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    models_dict[name] = m
    print(f"{name:<28s} | Trainable params: {n_params:,}")

print("\nModels created on CPU. Will move to GPU/MPS one by one during training.")

M1_Scratch_NoAttn            | Trainable params: 18,770,162
M2_Scratch_Attn              | Trainable params: 19,821,812
M3_Pretrained_NoAttn         | Trainable params: 22,062,834
M4_Pretrained_Attn           | Trainable params: 23,114,484

Models created on CPU. Will move to GPU/MPS one by one during training.


In [7]:
all_histories = {}

def _train_one(name):
    model = models_dict[name]
    
    # Dọn dẹp RAM/VRAM trước khi train
    gc.collect()
    if device.type == "mps": torch.mps.empty_cache()
    elif device.type == "cuda": torch.cuda.empty_cache()

    model.to(device)
    print(f"\n{'='*60}\n🚀 Bắt đầu huấn luyện: {name}\n{'='*60}")
    
    hist = train_model(
        model=model, name=name,
        train_loader=train_loader, val_loader=val_loader,
        answer_vocab=answer_vocab, device=device,
        epochs=cfg.train.epochs, lr=cfg.train.learning_rate,
        use_beam=False, beam_w=cfg.train.beam_width, ckpt_dir=cfg.ckpt_dir,
        label_smoothing=cfg.train.label_smoothing, patience=cfg.train.patience,
        grad_clip=cfg.train.grad_clip, tf_start=1.0, tf_end=cfg.train.tf_end,
        warmup_epochs=cfg.train.warmup_epochs, eval_every=cfg.train.eval_every,
        use_amp=False # Để False cho máy Mac
    )
    all_histories[name] = hist

    # Dọn dẹp RAM/VRAM sau khi train xong
    model.cpu()
    if device.type == "mps": torch.mps.empty_cache()
    elif device.type == "cuda": torch.cuda.empty_cache()
    gc.collect()
    print(f"✅ Đã giải phóng bộ nhớ cho {name}.\n")
    return hist

In [8]:
_train_one("M1_Scratch_NoAttn")


🚀 Bắt đầu huấn luyện: M1_Scratch_NoAttn


[M1_Scratch_NoAttn] Ep 1/5 tf=1.00:   0%|          | 0/2719 [00:00<?, ?it/s]

  Ep  1 loss=5.913/7.076 F1=0.139 B4=0.014 lr=1.0e-04
    ★ Saved best (F1=0.1391)


[M1_Scratch_NoAttn] Ep 2/5 tf=0.90:   0%|          | 0/2719 [00:00<?, ?it/s]

  Ep  2 loss=5.448/6.658 F1=0.148 B4=0.015 lr=2.0e-04
    ★ Saved best (F1=0.1476)


...

  Ep  3 loss=5.298/6.524 F1=0.154 B4=0.013 lr=3.0e-04
    ★ Saved best (F1=0.1538)


[M1_Scratch_NoAttn] Ep 4/5 tf=0.70:   0%|          | 0/2719 [00:00<?, ?it/s]

  Ep  4 loss=5.220/6.424 F1=0.163 B4=0.013 lr=1.5e-04
    ★ Saved best (F1=0.1628)


[M1_Scratch_NoAttn] Ep 5/5 tf=0.60:   0%|          | 0/2719 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
_train_one("M2_Scratch_Attn")

In [ ]:

_train_one("M3_Pretrained_NoAttn")

In [ ]:
_train_one("M4_Pretrained_Attn")


🚀 Bắt đầu huấn luyện: M4_Pretrained_Attn


[M4_Pretrained_Attn] Ep 1/5 tf=1.00:   0%|          | 0/2719 [00:00<?, ?it/s]

  Ep  1 loss=5.822/6.844 F1=0.141 B4=0.013 lr=1.0e-04
    ★ Saved best (F1=0.1411)


[M4_Pretrained_Attn] Ep 2/5 tf=0.90:   0%|          | 0/2719 [00:00<?, ?it/s]

  Ep  2 loss=5.326/6.592 F1=0.158 B4=0.015 lr=2.0e-04
    ★ Saved best (F1=0.1579)


[M4_Pretrained_Attn] Ep 3/5 tf=0.80:   0%|          | 0/2719 [00:00<?, ?it/s]

  Ep  3 loss=5.169/6.465 F1=0.162 B4=0.013 lr=3.0e-04
    ★ Saved best (F1=0.1620)


[M4_Pretrained_Attn] Ep 4/5 tf=0.70:   0%|          | 0/2719 [00:00<?, ?it/s]

  Ep  4 loss=5.061/6.400 F1=0.171 B4=0.016 lr=1.5e-04
    ★ Saved best (F1=0.1711)


[M4_Pretrained_Attn] Ep 5/5 tf=0.60:   0%|          | 0/2719 [00:00<?, ?it/s]

In [ ]:
test_results = {}
all_eval_data = {}

for name, model in models_dict.items():
    model.to(device)
    eval_data = evaluate_model(
        model=model, test_loader=test_loader,
        answer_vocab=answer_vocab, question_vocab=question_vocab,
        device=device, ckpt_dir=cfg.ckpt_dir, name=name,
        beam_width=cfg.train.beam_width,
    )
    test_results[name] = eval_data["metrics"]
    all_eval_data[name] = eval_data
    
    model.cpu()
    if device.type == "mps": torch.mps.empty_cache()
    elif device.type == "cuda": torch.cuda.empty_cache()

# In bảng so sánh
print("\n" + "=" * 110)
print(f"{'Model':<30s} {'Acc':>8s} {'EM':>8s} {'F1':>8s} {'METEOR':>8s} {'B-1':>8s} {'B-2':>8s} {'B-3':>8s} {'B-4':>8s}")
print("-" * 110)
for name, m in test_results.items():
    print(f"{name:<30s} {m['accuracy']:>8.4f} {m['em']:>8.4f} {m['f1']:>8.4f} {m['meteor']:>8.4f} {m['bleu1']:>8.4f} {m['bleu2']:>8.4f} {m['bleu3']:>8.4f} {m['bleu4']:>8.4f}")
print("=" * 110)

best_name = max(test_results, key=lambda k: test_results[k]["f1"])
print(f"\n★ Mô hình xuất sắc nhất: {best_name} (F1 = {test_results[best_name]['f1']:.4f})")

In [ ]:
# 1. Training curves (Loss, LR, Metrics)
plot_training_curves(all_histories, save_prefix="fig")

# 2. Radar chart & Bar chart
plot_radar_chart(test_results, save_path="fig4_radar.png")
plot_bar_chart(test_results, save_path="fig5_bar.png")

# 3. Phân tích loại câu hỏi
best_data = all_eval_data[best_name]
qtype_results = evaluate_by_question_type(best_data["preds"], best_data["refs"], best_data["questions"])
plot_question_type_analysis(qtype_results, save_path="fig9_question_type_analysis.png")
plot_confusion_matrix(best_data["preds"], best_data["refs"], best_data["questions"], save_path="fig10_confusion_matrix.png")

# 4. Vẽ Heatmap Spatial Attention MỚI
attn_model_name = "M4_Pretrained_Attn" if "M4_Pretrained_Attn" in models_dict else "M2_Scratch_Attn"
models_dict[attn_model_name].to(device)

visualize_attention(
    models_dict[attn_model_name], test_loader,
    answer_vocab, question_vocab, device,
    n=3, save_path="fig6_attention_heatmap.png"
)
visualize_attention_overlay(
    models_dict[attn_model_name], test_loader,
    answer_vocab, question_vocab, device,
    n=3, save_path="fig7_attention_overlay.png"
)

models_dict[attn_model_name].cpu()

In [ ]:
import urllib.request
from PIL import Image

def predict_custom_sample(model, image_path_or_url, question_text, device, q_vocab, a_vocab, transform):
    """
    Hàm hỗ trợ dự đoán câu trả lời cho một ảnh và câu hỏi bất kỳ.
    """
    model.eval()
    
    # 1. Tải ảnh từ URL hoặc đường dẫn local
    try:
        if image_path_or_url.startswith('http'):
            req = urllib.request.urlopen(image_path_or_url)
            image = Image.open(req).convert('RGB')
        else:
            image = Image.open(image_path_or_url).convert('RGB')
    except Exception as e:
        print(f"Lỗi không thể tải ảnh: {e}")
        return
        
    # Hiển thị ảnh
    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f"Q: {question_text}", fontsize=12, fontweight="bold")
    plt.show()
    
    # 2. Tiền xử lý ảnh (Resize, Normalize) -> tensor (1, 3, 224, 224)
    img_tensor = transform(image).unsqueeze(0).to(device)
    
    # 3. Tiền xử lý câu hỏi (Tokenize) -> tensor (1, T)
    q_tokens = [q_vocab.stoi["<SOS>"]] + q_vocab.numericalize(question_text) + [q_vocab.stoi["<EOS>"]]
    q_tensor = torch.tensor(q_tokens).unsqueeze(0).to(device)
    q_len = torch.tensor([len(q_tokens)]) # Giữ length ở CPU
    
    # 4. Sinh câu trả lời (Sử dụng Beam Search để có kết quả tốt nhất)
    with torch.no_grad():
        gen_out = model.generate(
            img_tensor, q_tensor, q_len, 
            use_beam=True, beam_width=5 
        )
        
    # 5. Giải mã output tensor thành text
    answer = decode_sequence(gen_out[0].cpu().tolist(), a_vocab)
    
    print("-" * 50)
    print(f"🤖 Dự đoán của mô hình: {answer}")
    print("-" * 50)
    
    return answer

# =====================================================================
# THỰC THI DEMO TRỰC TIẾP
# =====================================================================

# Lấy mô hình có điểm F1 cao nhất từ bước đánh giá trước đó
model_to_use = models_dict[best_name]
model_to_use.to(device)
print(f"Sử dụng mô hình xuất sắc nhất: {best_name}")

# --- BẠN CÓ THỂ THAY ĐỔI URL ẢNH VÀ CÂU HỎI Ở ĐÂY ---
test_url = "https://images.unsplash.com/photo-1543466835-00a7907e9de1?ixlib=rb-4.0.3&auto=format&fit=crop&w=800&q=80"
test_question = "what animal is in the picture and what color is it?"

# Chạy dự đoán
predict_custom_sample(
    model=model_to_use, 
    image_path_or_url=test_url, 
    question_text=test_question, 
    device=device, 
    q_vocab=question_vocab, 
    a_vocab=answer_vocab, 
    transform=transform_eval
)

# Giải phóng bộ nhớ sau khi dự đoán
model_to_use.cpu()
if device.type == "mps": torch.mps.empty_cache()
elif device.type == "cuda": torch.cuda.empty_cache()